# MedGemma - report processing

Builds the report text for **both** datasets in one canonical layout:
reformats the hand-written **DMID** reports (sections 1-6) and synthesizes
DMID-style reports for **VinDr-Mammo** from its label CSVs (final section).

Layout: `Breast Composition: <sentence> (ACR X).` / `BI-RADS: <sorted values>` /
`Findings:` bullets. Outputs go to `dmid/reports-processed/` and
`vindr-mammo/reports/`, which the split notebook reads.

In [ ]:
import ast
import re
from collections import Counter
from collections.abc import Iterable
from pathlib import Path

import pandas as pd

In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
ROOT_DIR = Path("/content/drive/MyDrive/MedGemma2026/main")
DATA_DIR = ROOT_DIR / "data"

# DMID
DMID_REPORTS_ORIGINAL  = DATA_DIR / "dmid" / "reports-original"      # input: hand-written DMID reports
DMID_REPORTS_PROCESSED = DATA_DIR / "dmid" / "reports-processed"    # output: reformatted DMID reports

# VinDr-Mammo
VINDR_BASE         = DATA_DIR / "vindr-mammo"
VINDR_BREAST_CSV   = VINDR_BASE / "breast-level_annotations.csv"
VINDR_FINDINGS_CSV = VINDR_BASE / "finding_annotations.csv"
VINDR_REPORTS_OUT  = VINDR_BASE / "reports"                        # output: synthesized VinDr reports

In [ ]:
# DMID reformatter (functions prefixed dmid_ / _*)
DMID_DENSITY_SENTENCE = {
    "A": "The breasts are almost entirely fatty",
    "B": "There are scattered areas of fibroglandular density",
    "C": "The breasts are heterogeneously dense, which may obscure small masses",
    "D": "The breasts are extremely dense, which lowers the sensitivity of mammography",
}

# lenient section-header patterns (tolerate the real typos in the corpus)
_H_COMPOSITION = re.compile(
    r"^\s*m?\s*(?:breast|brest)\s*compositions?\s*:", re.IGNORECASE
)  # + "Breast Compositions", "Brest", "mBreast"
_H_BIRADS = re.compile(r"^\s*bi[-\s]?rads\s*:", re.IGNORECASE)
_H_FINDINGS = re.compile(
    r"^\s*f[iu]n[a-z]*\s*:", re.IGNORECASE
)  # findings / fundings / finidngs / finding

# ACR letter anywhere in the report (tolerate missing paren / separators).
_ACR_RE = re.compile(r"\(?\s*ACR[\s:_-]*([A-Da-d])\b", re.IGNORECASE)

# A single BI-RADS token: 0-6 with optional a/b/c sub-category.
_BIRADS_TOKEN_RE = re.compile(r"[0-6][abc]?", re.IGNORECASE)
# Inline BI-RADS tag in any DMID spelling: "BIRADS-5", "(BIRADS 3)", "BI-RADS 4a".
_INLINE_BIRADS_RE = re.compile(r"\(?BI[-\s]?RADS[\s:-]*([0-6][abc]?)\)?", re.IGNORECASE)
_COMPLETE_TAIL_RE = re.compile(r"BI[-\s]?RADS[\s:-]*[0-6][abc]?$", re.IGNORECASE)
# A line that is ONLY a BI-RADS tag (e.g. a wrapped "(BIRADS 3)." on its own
# line) is never a finding by itself -- it always tags the preceding sentence.
_BARE_TAG_RE = re.compile(r"^\(?\s*BI[-\s]?RADS[\s:-]*[0-6][abc]?\s*\)?\.?$", re.IGNORECASE)
_ASYM_RE = re.compile(r"asymmetr", re.IGNORECASE)


def _classify_header(line: str):
    if _H_COMPOSITION.match(line):
        return "composition"
    if _H_BIRADS.match(line):
        return "birads"
    if _H_FINDINGS.match(line):
        return "findings"
    return None


def _strip_header(line: str) -> str:
    for pat in (_H_COMPOSITION, _H_BIRADS, _H_FINDINGS):
        if pat.match(line):
            return pat.sub("", line, count=1).strip()
    return line


def _parse_sections(text: str):
    """Assign every line to composition / birads / findings, order-independent."""
    comp, birads, find = [], [], []
    cur = None
    for ln in text.splitlines():
        h = _classify_header(ln)
        if h:
            cur = h
            inline = _strip_header(ln)
            if inline:
                {"composition": comp, "birads": birads, "findings": find}[h].append(inline)
        elif cur == "composition":
            comp.append(ln)
        elif cur == "birads":
            birads.append(ln)
        elif cur == "findings":
            find.append(ln)
        # lines before any header are dropped (none observed in the corpus)
    return comp, birads, find


def _split_birads_value(birads_lines):
    """First token-bearing line = header value; any further lines are findings
    that were written under the BIRADS line with no 'Findings:' header."""
    value, spill = "", []
    for bl in birads_lines:
        if not value and re.search(r"[0-6]", bl):
            value = bl
        elif value and bl.strip():
            spill.append(bl)
    return value, spill


# Breast Composition
def _composition(text: str, comp_lines):
    m = _ACR_RE.search(text)
    if m:
        letter = m.group(1).upper()
        return f"Breast Composition: {DMID_DENSITY_SENTENCE[letter]} (ACR {letter}).", letter
    desc = re.sub(r"\s+", " ", " ".join(l.strip() for l in comp_lines)).strip().rstrip(".")
    if not desc:
        desc = "could not be commented upon"
    return f"Breast Composition: {desc[0].upper() + desc[1:]}.", None


# BI-RADS header
def _sort_key(t: str):
    return (int(t[0]), t[1:].lower())


def _header_tokens(value_line: str):
    """BI-RADS tokens from the header value string (sorted, de-duplicated)."""
    return sorted(
        {m.group(0).lower() for m in _BIRADS_TOKEN_RE.finditer(value_line)}, key=_sort_key
    )


def _finding_tokens(find_lines):
    """BI-RADS tokens explicitly tagged on finding sentences (sorted, de-duplicated).
    Uses the inline-tag regex so stray digits in prose are not mistaken for BI-RADS."""
    joined = "\n".join(find_lines)
    return sorted({m.group(1).lower() for m in _INLINE_BIRADS_RE.finditer(joined)}, key=_sort_key)


def _merge_birads_header(header_tok, finding_tok):
    """The header carries the whole-case BI-RADS. If every header value is confirmed
    among the finding tags, adopt the union (folds in any finding-level BI-RADS the
    header omitted). Otherwise a header value is absent from the findings, which is a
    genuine contradiction, so keep the header unchanged: a single finding's BI-RADS
    may legitimately differ from the whole-case BI-RADS."""
    hs, fs = set(header_tok), set(finding_tok)
    merged = (
        sorted(hs | fs, key=_sort_key) if (fs and hs.issubset(fs)) else sorted(hs, key=_sort_key)
    )
    return merged


# Findings
def _looks_complete(line: str) -> bool:
    s = line.rstrip()
    if not s:
        return False
    if s.endswith((".", ")")):
        return True
    return bool(_COMPLETE_TAIL_RE.search(s))


def _reflow(raw_lines):
    """Merge soft-wrapped continuation lines back into whole sentences."""
    out, prev_trailing_ws = [], False
    for raw in raw_lines:
        if raw.strip() == "":
            continue
        curr = raw.strip()
        curr_leading_ws = raw[:1].isspace()
        # A standalone BI-RADS tag always attaches to the previous sentence,
        # even when that sentence already ended with a period.
        if out and _BARE_TAG_RE.match(curr):
            out[-1] = f"{out[-1].rstrip()} {curr}"
            prev_trailing_ws = raw[-1:].isspace()
            continue
        is_cont = False
        if out and not _looks_complete(out[-1]):
            starts_seen = bool(re.match(r"Seen\b", curr))
            starts_tag = bool(
                re.match(r"\(?\s*BI[-\s]?RADS", curr, re.IGNORECASE)
            )  # bare "(BIRADS 3)" tail
            starts_lower = curr[:1].islower()
            wrap_seam = prev_trailing_ws or curr_leading_ws
            if starts_seen or starts_tag or (wrap_seam and starts_lower):
                is_cont = True
        if is_cont:
            if curr.startswith("Seen "):
                curr = "seen " + curr[5:]
            out[-1] = f"{out[-1].rstrip()} {curr}"
        else:
            out.append(curr)
        prev_trailing_ws = raw[-1:].isspace()
    return out


def _split_sentences(reflowed: str):
    parts = re.split(r"(?<=\.)\s+(?=[A-Z])", reflowed.strip())
    return [p for p in parts if p.strip()]


def _normalize_inline_birads(text: str) -> str:
    out = _INLINE_BIRADS_RE.sub(lambda m: f"(BI-RADS {m.group(1).lower()})", text)
    out = re.sub(r"\s+", " ", out).strip()
    out = out.replace("( ", "(").replace(" )", ")")
    out = re.sub(
        r"\.\s*\(BI-RADS", " (BI-RADS", out
    )  # "lesion. (BI-RADS 3)" -> "lesion (BI-RADS 3)"
    out = re.sub(r"(?<=[^\s(])\(BI-RADS", " (BI-RADS", out)  # glued tag -> single space
    return out


def _bulletize(sentence: str) -> str:
    text = _normalize_inline_birads(sentence)
    text = re.sub(r"\s*\.+\s*$", "", text).strip()  # drop trailing period(s)
    text = re.sub(r"\s+", " ", text)
    if not text:
        return ""
    text = text[0].upper() + text[1:]
    return f"- {text}."


# User-approved canonical wording for recurring normal/benign findings
# Matched EXACTLY on the normalized finding sentence, so a longer finding that
# merely contains one of these phrases is left untouched. Values are lowercase;
# _bulletize re-capitalizes and adds the trailing period.
UNIFY_FINDINGS = True
_FINDING_CANON = {
    # 1. skin / nipple / pectoral (normal)
    "skin and nipple - no abnormality": "skin and nipple appear normal",
    "skin, nipple and pectoral muscle appear normal": "skin, nipple, and pectoral muscle appear normal",
    # 2. axillary node (normal / benign)
    "no significant axillary adenopathy seen": "no significant axillary adenopathy",
    "no significant axillary nodes seen": "no significant axillary adenopathy",
    "benign looking axillary adenopathy": "benign-looking axillary adenopathy",
    "benign-looking axillary node": "benign-looking axillary adenopathy",
    "few small benign looking axillary nodes seen": "few small benign-looking axillary nodes seen",
    # 3. negative: no soft opacity / microcalcification
    "no abnormal soft opacity or microcalcifications seen": "no abnormal soft opacity or microcalcification seen",
    "no abnormal soft opacity or microcalcification": "no abnormal soft opacity or microcalcification seen",
    "no soft opacity and microcalcification seen": "no abnormal soft opacity or microcalcification seen",
    "no abnormal microcalcification": "no abnormal microcalcification seen",
    "no microcalcification seen": "no abnormal microcalcification seen",
    "no microcalcifications seen": "no abnormal microcalcification seen",
    "no abnormal soft opacity": "no abnormal soft opacity seen",
    "no abnormal calcification": "no abnormal calcification seen",
    "no abnormal calcification seen within": "no abnormal calcification seen",
    "no abnormal calcification is seen within the lesion": "no abnormal calcification seen",
    # 4. benign / vascular calcification (present)
    "benign vascular calcifications seen": "benign vascular calcification seen",
    "benign-vascular calcification seen": "benign vascular calcification seen",
    "benign and vascular calcifications": "benign vascular calcification seen",
    "benign-looking calcifications seen": "benign-looking calcification seen",
    "benign-looking calcifications seen in the breast": "benign-looking calcification seen",
    "benign looking calcifications seen": "benign-looking calcification seen",
}


def _canon_key(sentence: str) -> str:
    """Same normalization used to survey the phrasings (lowercase, collapse
    whitespace, strip trailing period)."""
    s = re.sub(r"\s+", " ", sentence.strip().lower())
    return re.sub(r"[.\s]+$", "", s)


def _findings(find_lines):
    dropped_asym = unified = 0
    bullets = []
    for reflowed in _reflow(find_lines):
        for sent in _split_sentences(reflowed):
            if _ASYM_RE.search(sent):
                dropped_asym += 1
                continue
            key = _canon_key(sent)
            if UNIFY_FINDINGS and key in _FINDING_CANON:
                sent = _FINDING_CANON[key]
                unified += 1
            b = _bulletize(sent)
            if b:
                bullets.append(b)
    if not bullets:
        bullets.append("- No abnormal soft opacity.")
    return "Findings:\n" + "\n".join(bullets), dropped_asym, unified


# Assembly
def dmid_reformat_report(text: str):
    comp_lines, birads_lines, find_lines = _parse_sections(text)
    value_line, spill = _split_birads_value(birads_lines)
    find_lines = [l for l in (spill + find_lines) if _classify_header(l) is None]

    composition, acr = _composition(text, comp_lines)
    header_tok = _header_tokens(value_line)
    finding_tok = _finding_tokens(find_lines)
    merged = _merge_birads_header(header_tok, finding_tok)
    birads = "BI-RADS: " + ", ".join(merged)
    findings, dropped_asym, unified = _findings(find_lines)

    report = f"{composition}\n\n{birads}\n\n{findings}"
    meta = {
        "acr": acr,
        "birads_header": header_tok,  # values parsed from the BIRADS header line
        "birads_findings": finding_tok,  # values tagged inline on findings
        "birads_merged": merged,  # header actually written
        "birads_changed": merged != header_tok,  # findings added a value (rule 2)
        "birads_conflict": bool(finding_tok)
        and not set(header_tok).issubset(finding_tok),  # rule 3
        "dropped_asym": dropped_asym,
        "unified": unified,  # findings rewritten to canonical wording
    }
    return report, meta

In [ ]:
def dmid_convert_directory(in_dir: Path, out_dir: Path) -> int:
    out_dir.mkdir(parents=True, exist_ok=True)
    written = total_unified = 0
    dropped_no_acr, extended, conflicts, asym_files = [], [], [], []
    for txt_path in sorted(in_dir.glob("*.txt")):
        report, meta = dmid_reformat_report(txt_path.read_text(encoding="utf-8"))
        if meta["acr"] is None:  # rule 5: no ACR density -> drop
            dropped_no_acr.append(txt_path.name)
            continue
        (out_dir / txt_path.name).write_text(report, encoding="utf-8")
        written += 1
        total_unified += meta["unified"]
        if meta["birads_changed"]:
            extended.append((txt_path.name, meta["birads_header"], meta["birads_merged"]))
        if meta["birads_conflict"]:
            conflicts.append((txt_path.name, meta["birads_header"], meta["birads_findings"]))
        if meta["dropped_asym"]:
            asym_files.append(txt_path.name)

    print(f"Reformatted {written} DMID reports -> {out_dir}")
    print(f"Dropped (no ACR density): {len(dropped_no_acr)}  {dropped_no_acr}")
    print(f"Findings rewritten to canonical wording (rule 6): {total_unified}")
    print(f"BI-RADS header extended from findings (rule 2): {len(extended)}")
    for nm, h, m in extended:
        print(f"    {nm}: {h} -> {m}")
    print(f"BI-RADS header kept despite a finding tag (rule 3, contradiction): {len(conflicts)}")
    for nm, h, f in conflicts:
        print(f"    {nm}: header={h}  findings={f}")
    print(f"Reports with dropped asymmetry finding(s): {len(asym_files)}  {asym_files}")
    return written


n = dmid_convert_directory(DMID_REPORTS_ORIGINAL, DMID_REPORTS_PROCESSED)

Reformatted 508 DMID reports -> /content/drive/MyDrive/MedGemma2026/main/data/dmid/reports-processed
Dropped (no ACR density): 2  ['Img457.txt', 'Img458.txt']
Findings rewritten to canonical wording (rule 6): 226
BI-RADS header extended from findings (rule 2): 2
    Img061.txt: ['5'] -> ['3', '5']
    Img116.txt: ['4c'] -> ['4c', '5']
BI-RADS header kept despite a finding tag (rule 3, contradiction): 2
    Img023.txt: header=['4b']  findings=['4a']
    Img428.txt: header=['4c']  findings=['4b']
Reports with dropped asymmetry finding(s): 5  ['Img106.txt', 'Img125.txt', 'Img262.txt', 'Img272.txt', 'Img273.txt']


## Analysis 1 - BI-RADS header vs. BI-RADS tagged inside Findings

Some reports carry an inline `(BIRADS x)` tag on a finding sentence whose value is not in
the header. This flags every report where the header BI-RADS set differs from the set tagged
in the Findings, and shows the action taken:

* **extend header (rule 2)** - every header value is confirmed among the finding tags, so the
  finding-level value the header omitted is folded into the header (e.g. `5` -> `3, 5`).
* **keep header (rule 3)** - a header value is absent from the findings, i.e. a whole-case vs
  per-finding difference; the header is left unchanged and the per-finding tag stays inline.

In [ ]:
mismatches = []
for p in sorted(DMID_REPORTS_ORIGINAL.glob("*.txt")):
    _, birads_lines, find = _parse_sections(p.read_text(encoding="utf-8"))
    value, spill = _split_birads_value(birads_lines)
    find_lines = [l for l in (spill + find) if _classify_header(l) is None]
    header_tok = _header_tokens(value)
    finding_tok = _finding_tokens(find_lines)
    if finding_tok and set(finding_tok) != set(header_tok):
        merged = _merge_birads_header(header_tok, finding_tok)
        action = (
            "extend header (rule 2)"
            if merged != header_tok
            else "keep header (rule 3, contradiction)"
        )
        mismatches.append((p.name, header_tok, finding_tok, merged, action))

print(f"Reports where header BI-RADS != Findings BI-RADS: {len(mismatches)}")
for name, h, f, m, action in mismatches:
    print(f"  {name}: header={h}  findings={f}  ->  written={m}   [{action}]")

Reports where header BI-RADS != Findings BI-RADS: 4
  Img023.txt: header=['4b']  findings=['4a']  ->  written=['4b']   [keep header (rule 3, contradiction)]
  Img061.txt: header=['5']  findings=['3', '5']  ->  written=['3', '5']   [extend header (rule 2)]
  Img116.txt: header=['4c']  findings=['4c', '5']  ->  written=['4c', '5']   [extend header (rule 2)]
  Img428.txt: header=['4c']  findings=['4b']  ->  written=['4c']   [keep header (rule 3, contradiction)]


## Analysis 2 - finding groups with unified / near-unified phrasing

Groups of boilerplate normal / benign findings that recur with only trivial differences
(Oxford comma, hyphenation, singular vs plural, presence of a trailing `seen`). These are
candidates to collapse to one canonical wording for consistency across the corpus. Each
bucket prints its most common phrasings with counts.

In [ ]:
CATEGORIES = {
    "skin / nipple / pectoral (normal)": [r"\bskin\b", r"\bnipple\b", r"pectoral"],
    "axillary node (normal / benign)": [r"axill", r"adenopath", r"\bnode"],
    "negative: no soft opacity / microcalcification": [r"^no\b.*(opacit|calcif|microcalc)"],
    "benign / vascular calcification (present)": [r"(benign|vascular|popcorn|coarse).*calcif"],
}


def _norm(s):
    s = re.sub(r"\s+", " ", s.strip().lower())
    return re.sub(r"[.\s]+$", "", s)


phrases = {c: Counter() for c in CATEGORIES}
for p in sorted(DMID_REPORTS_ORIGINAL.glob("*.txt")):
    _, birads_lines, find = _parse_sections(p.read_text(encoding="utf-8"))
    _, spill = _split_birads_value(birads_lines)
    for reflowed in _reflow(spill + find):
        for sent in _split_sentences(reflowed):
            n = _norm(sent)
            for c, pats in CATEGORIES.items():
                if any(re.search(pt, n) for pt in pats):
                    phrases[c][n] += 1

for c, ph in phrases.items():
    print(f"\n=== {c}  ({sum(ph.values())} lines, {len(ph)} distinct phrasings) ===")
    for txt, k in ph.most_common(12):
        print(f"  {k:4d}  {txt}")


=== skin / nipple / pectoral (normal)  (541 lines, 93 distinct phrasings) ===
   200  skin and nipple appear normal
   164  skin, nipple, and pectoral muscle appear normal
    23  skin and nipple - no abnormality
    22  skin, nipple and pectoral muscle appear normal
    11  skin and pectoral muscle appear normal
     8  skin appears normal
     6  nipple appears normal
     4  pectoral muscle appears normal
     4  nipple retraction seen
     4  mild diffuse skin thickening seen
     3  mild skin thickening seen
     3  nipple and pectoral muscles appear normal

=== axillary node (normal / benign)  (259 lines, 69 distinct phrasings) ===
    51  no significant axillary adenopathy
    34  benign-looking axillary adenopathy
    24  no significant axillary adenopathy seen
    12  no axillary adenopathy
    11  small benign-looking axillary adenopathy
    11  few small benign-looking axillary nodes seen
     8  few small benign looking axillary nodes seen
     7  benign looking axillary a

## VinDr-Mammo report generation

VinDr-Mammo has no free-text reports, so DMID-style reports are **synthesized from the label
CSVs** (`breast-level_annotations.csv` + `finding_annotations.csv`). Aligned to DMID:

* Same **composition** density sentences and the same **`BI-RADS:`** header + sort order.
* **Findings scope** = every reported finding type EXCEPT the asymmetry family
  (`Focal Asymmetry`, `Asymmetry`, `Global Asymmetry`), which a single view cannot establish.
* Positive findings render as bullets. **Only mass and calcification** carry a `(BI-RADS x)`
  tail (when their per-finding BI-RADS is known); other findings carry no tail, and their
  BI-RADS are ignored (they do not feed the `BI-RADS:` header either -- only `breast_birads`
  plus mass/calc finding BI-RADS do). When none remain, the negative fallback
  `- No abnormal soft opacity.` is emitted (same as DMID).
* **Finding vocabulary aligned to DMID** (`mass` &rarr; `soft opacity`, `lymph node` &rarr;
  `axillary adenopathy`, calcifications &rarr; `microcalcifications`) and DMID's telegraphic
  `seen` with no copula (never `is`/`are`/`identified`/`present`/`noted`), e.g. `Soft
  opacity seen (BI-RADS 4c).`
* No case is dropped for finding reasons; density and breast BI-RADS are always present in
  VinDr, so there is no no-ACR drop here.

> Unlike DMID, VinDr reports list only positive findings plus the single negative line; the
> corpus has no annotation for normal structures (skin/nipple/axilla), so those normal
> statements that appear in DMID reports have no VinDr equivalent.

In [ ]:
# === VinDr-Mammo report generator (functions prefixed vindr_ / _vindr_*) =
VINDR_DENSITY_SENTENCE = {
    "A": "The breasts are almost entirely fatty",
    "B": "There are scattered areas of fibroglandular density",
    "C": "The breasts are heterogeneously dense, which may obscure small masses",
    "D": "The breasts are extremely dense, which lowers the sensitivity of mammography",
}

VINDR_KEEP_FINDINGS = {
    "Mass",
    "Suspicious Calcification",
    "Architectural Distortion",
    "Suspicious Lymph Node",
    "Skin Thickening",
    "Skin Retraction",
    "Nipple Retraction",
}
VINDR_DROPPED_CATEGORIES: set[str] = {"Focal Asymmetry", "Asymmetry", "Global Asymmetry"}

VINDR_SKIN_NIPPLE_PHRASE = {
    "Skin Thickening": "skin thickening",
    "Skin Retraction": "skin retraction",
    "Nipple Retraction": "nipple retraction",
}

# accepts both 'BIRADS:' and 'BI-RADS:' so the parsability check passes.
_VINDR_BIRADS_RE = re.compile(r"BI[-\s]?RADS:\s*(.+?)(?:\n|$)", re.IGNORECASEGNORECASE)
_VINDR_ACR_RE = re.compile(r"\(ACR[\s-]+([A-D])\s*\)", re.IGNORECASEGNORECASE)


def _vindr_birads_sort_key(token: str):
    """Numeric part first, then a/b/c sub-category -- identical to the DMID sort."""
    return (int(token[0]), token[1:].lower()) if token[:1].isdigit() else (99, token)


def _vindr_normalize_birads(raw: object) -> str:
    text = str(raw).strip()
    match = re.search(r"([0-6](?:[abc])?)", text, re.IGNORECASEGNORECASE)
    return match.group(1) if match else text


def _vindr_normalize_density(raw: object) -> str:
    text = str(raw).strip().upper()
    match = re.search(r"\b([A-D])\b", text)
    return match.group(1) if match else text


def _vindr_parse_categories(raw: object) -> list[str]:
    if isinstance(raw, list):
        return list(raw)
    try:
        parsed = ast.literal_eval(raw) if isinstance(raw, str) else []
    except (ValueError, SyntaxError):
        return []
    return list(parsed) if isinstance(parsed, list) else []


def _vindr_category_birads_for_image(findings_df: pd.DataFrame, image_id: str):
    result: dict[str, str | None] = {}
    _SENTINEL = "__unset__"
    for _, row in findings_df[findings_df["image_id"] == image_id].iterrows():
        raw_fb = row.get("finding_birads")
        b = None if pd.isna(raw_fb) else _vindr_normalize_birads(raw_fb)
        for category in _vindr_parse_categories(row.get("finding_categories")):
            if category == "No Finding" or category in VINDR_DROPPED_CATEGORIES:
                continue
            prev = result.get(category, _SENTINEL)
            # Unset -> take b; otherwise take b only if it is a known value that
            # is more severe than what is stored (or nothing is stored yet).
            if prev == _SENTINEL or (b is not None and (prev is None or b > prev)):
                result[category] = b
    return result


def _vindr_join_with_and(items: list[str]) -> str:
    if not items:
        return ""
    if len(items) == 1:
        return items[0]
    if len(items) == 2:
        return items[0] + " and " + items[1]
    return ", ".join(items[:-1]) + ", and " + items[-1]  # Oxford comma, matches DMID


def _vindr_composition_section(density_letter: str) -> str:
    sentence = VINDR_DENSITY_SENTENCE.get(density_letter, "The breasts are of intermediate density")
    return f"Breast Composition: {sentence} (ACR {density_letter})."


def _vindr_format_bullet(phrase: str, birads_value: str | None) -> str:
    if birads_value:
        return f"{phrase} (BI-RADS {birads_value})."
    return f"{phrase}."


# L2 vocabulary alignment with DMID: DMID says "soft opacity" (not "mass"),
# "axillary adenopathy" (not "lymph node"), "microcalcification", and uses the
# verb "seen" (never "identified"/"present"/"noted").
VINDR_BULLET_PHRASE = {
    "Mass": "Soft opacity seen",
    "Architectural Distortion": "Architectural distortion seen",
    "Suspicious Calcification": "Suspicious microcalcifications seen",
    "Suspicious Lymph Node": "Suspicious axillary adenopathy seen",
}
VINDR_FINDING_ORDER = (
    "Mass",
    "Architectural Distortion",
    "Suspicious Calcification",
    "Suspicious Lymph Node",
)
# Only mass and calcification carry a per-finding (BI-RADS x) tag -- mirrors DMID,
# where inline BI-RADS tags sit on the primary lesion, not on nodes / distortion.
VINDR_BIRADS_TAGGED = {"Mass", "Suspicious Calcification"}


def _vindr_findings_section(cats_to_birads: dict[str, str | None]) -> str:
    bullets: list[str] = []
    for cat in VINDR_FINDING_ORDER:
        if cat in cats_to_birads:
            tag = cats_to_birads[cat] if cat in VINDR_BIRADS_TAGGED else None
            bullets.append(_vindr_format_bullet(VINDR_BULLET_PHRASE[cat], tag))
    skin_nipple = [
        VINDR_SKIN_NIPPLE_PHRASE[c]
        for c in ("Skin Thickening", "Skin Retraction", "Nipple Retraction")
        if c in cats_to_birads
    ]
    if skin_nipple:
        phrase = _vindr_join_with_and(skin_nipple)
        phrase = phrase[0].upper() + phrase[1:]
        bullets.append(f"{phrase} seen.")
    if not bullets:
        bullets.append("No abnormal soft opacity.")
    body = "\n".join(f"- {line}" for line in bullets)
    return f"Findings:\n{body}"


def _vindr_birads_section(breast_birads: str, cats_to_birads: dict[str, str | None]) -> str:
    # Only the breast-level BI-RADS and the finding BI-RADS that are actually written
    # on the Findings (mass & calcification) count; other categories' BI-RADS are ignored.
    all_birads = sorted(
        {_vindr_normalize_birads(breast_birads)}
        | {b for c, b in cats_to_birads.items() if b and c in VINDR_BIRADS_TAGGED},
        key=_vindr_birads_sort_key,
    )
    return "BI-RADS: " + ", ".join(all_birads)


def _vindr_synthesize_report(*, breast_birads, breast_density, category_birads) -> str:
    density_letter = _vindr_normalize_density(breast_density)
    composition = _vindr_composition_section(density_letter)
    birads_line = _vindr_birads_section(breast_birads, category_birads)
    findings = _vindr_findings_section(category_birads)
    return f"{composition}\n\n{birads_line}\n\n{findings}"


def _vindr_assert_parsable(report_text: str) -> None:
    assert _VINDR_BIRADS_RE.search(report_text), f"BI-RADS regex failed on:\n{report_text}"
    assert _VINDR_ACR_RE.search(report_text), f"ACR regex failed on:\n{report_text}"


def vindr_convert_rows(
    breast_df, findings_df, out_reports: Path, *, keep_only_birads: Iterable[str] | None = None
) -> int:
    out_reports.mkdir(parents=True, exist_ok=True)
    keep = set(keep_only_birads) if keep_only_birads is not None else None
    written = 0
    findings_by_image = dict(list(findings_df.groupby("image_id")))
    for _, row in breast_df.iterrows():
        breast_birads = int(row["breast_birads"].strip().split()[-1])
        if keep is not None and breast_birads not in keep:
            continue
        category_birads = _vindr_category_birads_for_image(
            findings_by_image[row["image_id"]], row["image_id"]
        )
        kept_birads = {c: b for c, b in category_birads.items() if c in VINDR_KEEP_FINDINGS}
        report = _vindr_synthesize_report(
            breast_birads=row["breast_birads"],
            breast_density=row["breast_density"],
            category_birads=kept_birads,
        )
        _vindr_assert_parsable(report)
        (out_reports / f"{row['image_id']}.txt").write_text(report, encoding="utf-8")
        written += 1
    return written

In [9]:
breast_df = pd.read_csv(VINDR_BREAST_CSV)
findings_df = pd.read_csv(VINDR_FINDINGS_CSV)
n = vindr_convert_rows(breast_df, findings_df, out_reports=VINDR_REPORTS_OUT)
print(f"Wrote {n} VinDr reports -> {VINDR_REPORTS_OUT}")

Wrote 20000 VinDr reports -> /content/drive/MyDrive/MedGemma2026/main/data/vindr-mammo/reports
